<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/quant_DL_ashu/notebooks/02_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports n Data read

In [1]:
from sklearn.preprocessing import StandardScaler
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json
import joblib

In [2]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# read the stored raw data

shared_project_dir = Path("/content/drive/MyDrive/Quant Research")
data_dir = shared_project_dir / "data"
raw_dir = data_dir / "raw"

filename = "crypto_binance_1d_btc_eth_sol_xrp_raw.parquet"

shared_file_path = (
    raw_dir /
    filename
)

drive_df = pd.read_parquet(shared_file_path)

print(drive_df.shape)


(11738, 13)


### Basic Check and PreProcessing

In [4]:
drive_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11738 entries, 0 to 11737
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   open_time               11738 non-null  datetime64[ns, UTC]
 1   close_time              11738 non-null  datetime64[ns, UTC]
 2   asset                   11738 non-null  object             
 3   symbol                  11738 non-null  object             
 4   open                    11738 non-null  float64            
 5   high                    11738 non-null  float64            
 6   low                     11738 non-null  float64            
 7   close                   11738 non-null  float64            
 8   base_volume             11738 non-null  float64            
 9   quote_volume            11738 non-null  float64            
 10  number_of_trades        11738 non-null  int64              
 11  taker_buy_base_volume   11738 non-null  f

In [5]:
drive_df.isna().sum()

,0
open_time,0
close_time,0
asset,0
symbol,0
open,0
high,0
low,0
close,0
base_volume,0
quote_volume,0


In [6]:
drive_df.head()

,open_time,close_time,asset,symbol,open,high,low,close,base_volume,quote_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2017-08-17 00:00:00+00:00,2017-08-17 23:59:59.999000+00:00,BTC,BTCUSDT,4261.48,4485.39,4200.74,4285.08,795.150377,3.454770e+06,3427,616.248541,2.678216e+06
1,2017-08-17 00:00:00+00:00,2017-08-17 23:59:59.999000+00:00,ETH,ETHUSDT,301.13,312.18,298.00,302.00,7030.710340,2.154655e+06,4522,6224.589990,1.908705e+06
2,2017-08-18 00:00:00+00:00,2017-08-18 23:59:59.999000+00:00,BTC,BTCUSDT,4285.08,4371.52,3938.77,4108.37,1199.888264,5.086958e+06,5233,972.868710,4.129123e+06
3,2017-08-18 00:00:00+00:00,2017-08-18 23:59:59.999000+00:00,ETH,ETHUSDT,302.00,311.79,283.94,293.96,9537.846460,2.858947e+06,5658,7452.435420,2.240813e+06
4,2017-08-19 00:00:00+00:00,2017-08-19 23:59:59.999000+00:00,BTC,BTCUSDT,4108.37,4184.69,3850.00,4139.98,381.309763,1.549484e+06,2153,274.336042,1.118002e+06


In [7]:
required_columns = [
    "open_time",
    "close_time",
    "asset",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
]

missing_columns = (
    set(required_columns)
    - set(drive_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Rows before validation:", len(drive_df))

drive_df = (
    drive_df
    .dropna(subset=required_columns)
    .copy()
)

valid_row_mask = (
    (drive_df["open"] > 0)
    & (drive_df["high"] > 0)
    & (drive_df["low"] > 0)
    & (drive_df["close"] > 0)
    & (drive_df["base_volume"] >= 0)
)

drive_df = (
    drive_df.loc[valid_row_mask]
    .sort_values(
        ["symbol", "open_time"]
    )
    .drop_duplicates(
        subset=["symbol", "open_time"],
        keep="last"
    )
    .reset_index(drop=True)
)

drive_df["date"] = (
    drive_df["open_time"].dt.floor("D")
)

print("Rows after validation:", len(drive_df))

display(
    drive_df.groupby("symbol").agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        rows=("date", "size"),
        unique_dates=("date", "nunique"),
    )
)

Rows before validation: 11738
Rows after validation: 11738


,first_date,last_date,rows,unique_dates
symbol,,,,
BTCUSDT,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272,3272
ETHUSDT,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272,3272
SOLUSDT,2020-08-11 00:00:00+00:00,2026-08-01 00:00:00+00:00,2182,2182
XRPUSDT,2018-05-04 00:00:00+00:00,2026-08-01 00:00:00+00:00,3012,3012


In [8]:
assert not drive_df.duplicated(
    ["symbol", "date"]
).any(), "Duplicate symbol-date rows found."

assert (
    drive_df["high"]
    >= drive_df[["open", "close", "low"]].max(axis=1)
).all()

assert (
    drive_df["low"]
    <= drive_df[["open", "close", "high"]].min(axis=1)
).all()

print("Basic validation passed.")

Basic validation passed.


In [9]:
# ============================================================
# DAILY RETURN AND VOLATILITY CONSTRUCTION
# ============================================================

daily_df = (
    drive_df
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
    .copy()
)

# Rename daily OHLCV columns for compatibility
# with the downstream modelling notebooks.

daily_df = daily_df.rename(
    columns={
        "open": "daily_open",
        "high": "daily_high",
        "low": "daily_low",
        "close": "daily_close",
        "base_volume": "daily_volume",
    }
)

In [10]:
previous_date = (
    daily_df
    .groupby("symbol")["date"]
    .shift(1)
)

previous_close = (
    daily_df
    .groupby("symbol")["daily_close"]
    .shift(1)
)

raw_daily_return = np.log(
    daily_df["daily_close"]
    / previous_close
)

# Do not calculate a return across a missing date.
previous_day_is_consecutive = (
    daily_df["date"] - previous_date
).eq(
    pd.Timedelta(days=1)
)

daily_df["daily_return"] = (
    raw_daily_return.where(
        previous_day_is_consecutive
    )
)

### Realized Variance Calculation

In [11]:
# Daily squared-return variance proxy
daily_df["realized_variance"] = (
    daily_df["daily_return"] ** 2
)

# Daily volatility proxy
daily_df["realized_volatility"] = (
    daily_df["daily_return"].abs()
)

EPSILON = 1e-12

daily_df["log_realized_variance"] = np.log(
    daily_df["realized_variance"]
    + EPSILON
)

In [12]:
# printing to check the columns

print(drive_df.columns, "\n")
print(daily_df.columns)

Index(['open_time', 'close_time', 'asset', 'symbol', 'open', 'high', 'low',
       'close', 'base_volume', 'quote_volume', 'number_of_trades',
       'taker_buy_base_volume', 'taker_buy_quote_volume', 'date'],
      dtype='object') 

Index(['open_time', 'close_time', 'asset', 'symbol', 'daily_open',
       'daily_high', 'daily_low', 'daily_close', 'daily_volume',
       'quote_volume', 'number_of_trades', 'taker_buy_base_volume',
       'taker_buy_quote_volume', 'date', 'daily_return', 'realized_variance',
       'realized_volatility', 'log_realized_variance'],
      dtype='object')


### Creating Initial features

In [13]:
daily_df["absolute_daily_return"] = (
    daily_df["daily_return"].abs()
)

daily_df["high_low_range"] = np.log(
    daily_df["daily_high"]
    / daily_df["daily_low"]
)

daily_df["log_volume"] = np.log1p(
    daily_df["daily_volume"]
)

In [14]:
grouped_rv = daily_df.groupby(
    "symbol"
)["realized_variance"]

daily_df["rv_mean_7d"] = grouped_rv.transform(
    lambda x: x.rolling(
        window=7,
        min_periods=7
    ).mean()
)

daily_df["rv_mean_30d"] = grouped_rv.transform(
    lambda x: x.rolling(
        window=30,
        min_periods=30
    ).mean()
)

In [15]:
date_6_rows_ago = (
    daily_df.groupby("symbol")["date"]
    .shift(6)
)

date_29_rows_ago = (
    daily_df.groupby("symbol")["date"]
    .shift(29)
)

valid_7d_window = (
    daily_df["date"] - date_6_rows_ago
).eq(pd.Timedelta(days=6))

valid_30d_window = (
    daily_df["date"] - date_29_rows_ago
).eq(pd.Timedelta(days=29))

daily_df.loc[
    ~valid_7d_window,
    "rv_mean_7d"
] = np.nan

daily_df.loc[
    ~valid_30d_window,
    "rv_mean_30d"
] = np.nan

In [16]:
# Build an exact next-calendar-day target lookup.

target_lookup = (
    daily_df[
        [
            "symbol",
            "date",
            "log_realized_variance",
            "realized_variance",
        ]
    ]
    .rename(
        columns={
            "date": "target_date",
            "log_realized_variance": "target_log_rv",
            "realized_variance": "target_rv",
        }
    )
)

daily_df["target_date"] = (
    daily_df["date"]
    + pd.Timedelta(days=1)
)

daily_df = daily_df.merge(
    target_lookup,
    on=["symbol", "target_date"],
    how="left",
    validate="one_to_one"
)

In [17]:
available_targets = daily_df.loc[
    daily_df["target_log_rv"].notna()
].copy()

assert (
    available_targets["target_date"]
    - available_targets["date"]
).eq(pd.Timedelta(days=1)).all()

print(
    "Rows with valid next-day target:",
    len(available_targets)
)

print(
    "Rows without a next-day target:",
    daily_df["target_log_rv"].isna().sum()
)

Rows with valid next-day target: 11734
Rows without a next-day target: 4


In [18]:
daily_df.head()

,open_time,close_time,asset,symbol,daily_open,daily_high,daily_low,daily_close,daily_volume,quote_volume,...,realized_volatility,log_realized_variance,absolute_daily_return,high_low_range,log_volume,rv_mean_7d,rv_mean_30d,target_date,target_log_rv,target_rv
0,2017-08-17 00:00:00+00:00,2017-08-17 23:59:59.999000+00:00,BTC,BTCUSDT,4261.48,4485.39,4200.74,4285.08,795.150377,3.454770e+06,...,NaN,NaN,NaN,0.065565,6.679788,NaN,NaN,2017-08-18 00:00:00+00:00,-6.334804,0.001773
1,2017-08-18 00:00:00+00:00,2017-08-18 23:59:59.999000+00:00,BTC,BTCUSDT,4285.08,4371.52,3938.77,4108.37,1199.888264,5.086958e+06,...,0.042113,-6.334804,0.042113,0.104242,7.090817,NaN,NaN,2017-08-19 00:00:00+00:00,-9.742286,0.000059
2,2017-08-19 00:00:00+00:00,2017-08-19 23:59:59.999000+00:00,BTC,BTCUSDT,4108.37,4184.69,3850.00,4139.98,381.309763,1.549484e+06,...,0.007665,-9.742286,0.007665,0.083359,5.946231,NaN,NaN,2017-08-20 00:00:00+00:00,-8.677400,0.000170
3,2017-08-20 00:00:00+00:00,2017-08-20 23:59:59.999000+00:00,BTC,BTCUSDT,4120.98,4211.08,4032.62,4086.29,467.083022,1.930364e+06,...,0.013053,-8.677400,0.013053,0.043303,6.148646,NaN,NaN,2017-08-21 00:00:00+00:00,-8.108200,0.000301
4,2017-08-21 00:00:00+00:00,2017-08-21 23:59:59.999000+00:00,BTC,BTCUSDT,4069.13,4119.62,3911.79,4016.00,691.743060,2.797232e+06,...,0.017351,-8.108200,0.017351,0.051766,6.540659,NaN,NaN,2017-08-22 00:00:00+00:00,-10.245937,0.000036


### Assigning train/val/test period

In [19]:
TRAIN_END = pd.Timestamp(
    "2024-07-31",
    tz="UTC"
)

VALIDATION_START = pd.Timestamp(
    "2024-08-01",
    tz="UTC"
)

VALIDATION_END = pd.Timestamp(
    "2025-07-31",
    tz="UTC"
)

TEST_START = pd.Timestamp(
    "2025-08-01",
    tz="UTC"
)

TEST_END = pd.Timestamp(
    "2026-07-31",
    tz="UTC"
)

In [20]:
def assign_split(target_date):

    if pd.isna(target_date):
        return None

    if target_date <= TRAIN_END:
        return "train"

    if VALIDATION_START <= target_date <= VALIDATION_END:
        return "validation"

    if TEST_START <= target_date <= TEST_END:
        return "test"

    return "outside"


daily_df["split"] = (
    daily_df["target_date"]
    .apply(assign_split)
)

In [21]:
daily_df.groupby("split").agg(
    start_target_date=("target_date", "min"),
    end_target_date=("target_date", "max"),
    number_of_rows=("target_date", "count"),
)

,start_target_date,end_target_date,number_of_rows
split,,,
outside,2026-08-01 00:00:00+00:00,2026-08-02 00:00:00+00:00,8
test,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00,1460
train,2017-08-18 00:00:00+00:00,2024-07-31 00:00:00+00:00,8810
validation,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00,1460


In [22]:
print("Final daily feature shape:")
print(daily_df.shape)

display(
    daily_df.groupby("symbol").agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observations=("date", "size"),
        valid_returns=(
            "daily_return",
            "count"
        ),
        valid_targets=(
            "target_log_rv",
            "count"
        ),
    )
)

display(
    daily_df[
        [
            "symbol",
            "date",
            "daily_close",
            "daily_return",
            "realized_variance",
            "realized_volatility",
            "log_realized_variance",
            "rv_mean_7d",
            "rv_mean_30d",
            "target_date",
            "target_log_rv",
            "split",
        ]
    ].head(35)
)

print(
    daily_df[
        [
            "daily_return",
            "realized_variance",
            "realized_volatility",
            "log_realized_variance",
            "rv_mean_7d",
            "rv_mean_30d",
            "target_log_rv",
        ]
    ].isna().sum()
)

Final daily feature shape:
(11738, 27)


,first_date,last_date,observations,valid_returns,valid_targets
symbol,,,,,
BTCUSDT,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272,3271,3271
ETHUSDT,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272,3271,3271
SOLUSDT,2020-08-11 00:00:00+00:00,2026-08-01 00:00:00+00:00,2182,2181,2181
XRPUSDT,2018-05-04 00:00:00+00:00,2026-08-01 00:00:00+00:00,3012,3011,3011


,symbol,date,daily_close,daily_return,realized_variance,realized_volatility,log_realized_variance,rv_mean_7d,rv_mean_30d,target_date,target_log_rv,split
0,BTCUSDT,2017-08-17 00:00:00+00:00,4285.08,NaN,NaN,NaN,NaN,NaN,NaN,2017-08-18 00:00:00+00:00,-6.334804,train
1,BTCUSDT,2017-08-18 00:00:00+00:00,4108.37,-0.042113,0.001773,0.042113,-6.334804,NaN,NaN,2017-08-19 00:00:00+00:00,-9.742286,train
2,BTCUSDT,2017-08-19 00:00:00+00:00,4139.98,0.007665,0.000059,0.007665,-9.742286,NaN,NaN,2017-08-20 00:00:00+00:00,-8.677400,train
3,BTCUSDT,2017-08-20 00:00:00+00:00,4086.29,-0.013053,0.000170,0.013053,-8.677400,NaN,NaN,2017-08-21 00:00:00+00:00,-8.108200,train
4,BTCUSDT,2017-08-21 00:00:00+00:00,4016.00,-0.017351,0.000301,0.017351,-8.108200,NaN,NaN,2017-08-22 00:00:00+00:00,-10.245937,train
5,BTCUSDT,2017-08-22 00:00:00+00:00,4040.00,0.005958,0.000036,0.005958,-10.245937,NaN,NaN,2017-08-23 00:00:00+00:00,-8.017780,train
6,BTCUSDT,2017-08-23 00:00:00+00:00,4114.01,0.018154,0.000330,0.018154,-8.017780,NaN,NaN,2017-08-24 00:00:00+00:00,-6.075896,train
7,BTCUSDT,2017-08-24 00:00:00+00:00,4316.01,0.047933,0.002298,0.047933,-6.075896,0.000709,NaN,2017-08-25 00:00:00+00:00,-9.602494,train
8,BTCUSDT,2017-08-25 00:00:00+00:00,4280.68,-0.008219,0.000068,0.008219,-9.602494,0.000466,NaN,2017-08-26 00:00:00+00:00,-8.659257,train
9,BTCUSDT,2017-08-26 00:00:00+00:00,4337.44,0.013172,0.000174,0.013172,-8.659257,0.000482,NaN,2017-08-27 00:00:00+00:00,-10.120464,train


daily_return               4
realized_variance          4
realized_volatility        4
log_realized_variance      4
rv_mean_7d                28
rv_mean_30d              120
target_log_rv              4
dtype: int64


In [23]:
print(daily_df.columns)

FEATURE_COLUMNS = [
    "log_realized_variance",
    "daily_return",
    "absolute_daily_return",
    "high_low_range",
    "log_volume",
    "rv_mean_7d",
    "rv_mean_30d",
]

Index(['open_time', 'close_time', 'asset', 'symbol', 'daily_open',
       'daily_high', 'daily_low', 'daily_close', 'daily_volume',
       'quote_volume', 'number_of_trades', 'taker_buy_base_volume',
       'taker_buy_quote_volume', 'date', 'daily_return', 'realized_variance',
       'realized_volatility', 'log_realized_variance', 'absolute_daily_return',
       'high_low_range', 'log_volume', 'rv_mean_7d', 'rv_mean_30d',
       'target_date', 'target_log_rv', 'target_rv', 'split'],
      dtype='object')


In [24]:
### Date - Continuity Check

def create_sequences(asset_df, feature_columns, lookback_days=30):

    asset_df = (asset_df.sort_values("date").reset_index(drop=True).copy())

    X = []
    y = []
    metadata = []

    feature_values = asset_df[feature_columns].to_numpy(dtype=np.float32)

    target_values = asset_df["target_log_rv"].to_numpy(dtype=np.float32)

    for end_idx in range(lookback_days - 1, len(asset_df)):

        start_idx = (
            end_idx - lookback_days + 1
        )

        sequence = feature_values[
            start_idx:end_idx + 1
        ]

        target = target_values[end_idx]

        sequence_dates = asset_df.loc[
            start_idx:end_idx,
            "date"
        ]

        # Input must contain lookback_days consecutive calendar days.
        date_differences = (
            sequence_dates.diff().dropna()
        )

        if not date_differences.eq(
            pd.Timedelta(days=1)
        ).all():
            continue

        if np.isnan(sequence).any():
            continue

        if np.isnan(target):
            continue

        sequence_end_date = asset_df.loc[
            end_idx,
            "date"
        ]

        target_date = asset_df.loc[
            end_idx,
            "target_date"
        ]

        # Target must be exactly the following calendar day.
        if (
            target_date - sequence_end_date
            != pd.Timedelta(days=1)
        ):
            continue

        split = asset_df.loc[
            end_idx,
            "split"
        ]

        if split not in {
            "train",
            "validation",
            "test"
        }:
            continue

        X.append(sequence)
        y.append(target)

        metadata.append({
            "symbol": asset_df.loc[
                end_idx,
                "symbol"
            ],
            "sequence_start_date": asset_df.loc[
                start_idx,
                "date"
            ],
            "sequence_end_date": sequence_end_date,
            "target_date": target_date,
            "split": split,
            "actual_rv": asset_df.loc[
                end_idx,
                "target_rv"
            ],
        })

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(metadata)
    )

In [25]:
LOOKBACK_OPTIONS = [7, 14, 30]

btc_df = daily_df.loc[
    daily_df["symbol"] == "BTCUSDT"
].copy()

print("BTC daily rows:", btc_df.shape)

BTC daily rows: (3272, 27)


In [26]:
def prepare_lookback_dataset(asset_df, feature_columns, lookback_days):

    X, y, sequence_metadata = create_sequences(
        asset_df=asset_df,
        feature_columns=feature_columns,
        lookback_days=lookback_days,
    )

    train_mask = (
        sequence_metadata["split"] == "train"
    ).to_numpy()

    validation_mask = (
        sequence_metadata["split"] == "validation"
    ).to_numpy()

    test_mask = (
        sequence_metadata["split"] == "test"
    ).to_numpy()

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_val = X[validation_mask]
    y_val = y[validation_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    if len(X_train) == 0:
      raise ValueError(
          f"No training sequences found for "
          f"{lookback_days}-day lookback."
      )

    if len(X_val) == 0:
        raise ValueError(
            f"No validation sequences found for "
            f"{lookback_days}-day lookback."
        )

    if len(X_test) == 0:
        raise ValueError(
            f"No test sequences found for "
            f"{lookback_days}-day lookback."
        )

    train_metadata = (
        sequence_metadata.loc[train_mask]
        .reset_index(drop=True)
    )

    validation_metadata = (
        sequence_metadata.loc[validation_mask]
        .reset_index(drop=True)
    )

    test_metadata = (
        sequence_metadata.loc[test_mask]
        .reset_index(drop=True)
    )

    assert len(X_train) == len(y_train) == len(train_metadata)
    assert len(X_val) == len(y_val) == len(validation_metadata)
    assert len(X_test) == len(y_test) == len(test_metadata)

    assert X_train.shape[1] == lookback_days
    assert X_val.shape[1] == lookback_days
    assert X_test.shape[1] == lookback_days

    n_features = X_train.shape[2]

    scaler = StandardScaler()

    scaler.fit(
        X_train.reshape(-1, n_features)
    )

    X_train_scaled = scaler.transform(
        X_train.reshape(-1, n_features)
    ).reshape(X_train.shape)

    X_val_scaled = scaler.transform(
        X_val.reshape(-1, n_features)
    ).reshape(X_val.shape)

    X_test_scaled = scaler.transform(
        X_test.reshape(-1, n_features)
    ).reshape(X_test.shape)

    assert (
        sequence_metadata["target_date"]
        - sequence_metadata["sequence_end_date"]
    ).eq(pd.Timedelta(days=1)).all()

    return {
        "X_train_raw": X_train,
        "X_val_raw": X_val,
        "X_test_raw": X_test,
        "X_train_scaled": X_train_scaled,
        "X_val_scaled": X_val_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "train_metadata": train_metadata,
        "validation_metadata": validation_metadata,
        "test_metadata": test_metadata,
        "sequence_metadata": sequence_metadata,
        "scaler": scaler,
    }

In [27]:
lookback_datasets = {}

for lookback_days in LOOKBACK_OPTIONS:
    dataset = prepare_lookback_dataset(
        asset_df=btc_df,
        feature_columns=FEATURE_COLUMNS,
        lookback_days=lookback_days,
    )

    lookback_datasets[lookback_days] = dataset

    print(
        f"\nLookback: {lookback_days} days"
    )

    print(
        "Train:",
        dataset["X_train_scaled"].shape
    )

    print(
        "Validation:",
        dataset["X_val_scaled"].shape
    )

    print(
        "Test:",
        dataset["X_test_scaled"].shape
    )

assert set(lookback_datasets.keys()) == {7, 14, 30}


Lookback: 7 days
Train: (2504, 7, 7)
Validation: (365, 7, 7)
Test: (365, 7, 7)

Lookback: 14 days
Train: (2497, 14, 7)
Validation: (365, 14, 7)
Test: (365, 14, 7)

Lookback: 30 days
Train: (2481, 30, 7)
Validation: (365, 30, 7)
Test: (365, 30, 7)


In [28]:
sequence_count_rows = []

for lookback_days, dataset in (
    lookback_datasets.items()
):
    sequence_count_rows.append({
        "lookback_days": lookback_days,
        "train_sequences": (
            dataset["X_train_scaled"].shape[0]
        ),
        "validation_sequences": (
            dataset["X_val_scaled"].shape[0]
        ),
        "test_sequences": (
            dataset["X_test_scaled"].shape[0]
        ),
    })

sequence_count_df = pd.DataFrame(
    sequence_count_rows
)

sequence_count_df = (
    pd.DataFrame(sequence_count_rows)
    .sort_values("lookback_days")
    .reset_index(drop=True)
)

display(sequence_count_df)

,lookback_days,train_sequences,validation_sequences,test_sequences
0,7,2504,365,365
1,14,2497,365,365
2,30,2481,365,365


In [29]:
model_data_root = (
    shared_project_dir
    / "data"
    / "model_ready"
)

model_data_root.mkdir(
    parents=True,
    exist_ok=True
)

daily_data_path = (
    model_data_root
    / "crypto_daily_features_v1.parquet"
)

daily_df.to_parquet(
    daily_data_path,
    index=False
)

print(
    "Saved daily feature table to:",
    daily_data_path
)

Saved daily feature table to: /content/drive/MyDrive/Quant Research/data/model_ready/crypto_daily_features_v1.parquet


In [30]:
model_ready_dir = (
    shared_project_dir
    / "data"
    / "model_ready"
    / "BTCUSDT"
)

model_ready_dir.mkdir(
    parents=True,
    exist_ok=True
)

sequence_count_df.to_csv(
    model_ready_dir
    / "lookback_sequence_counts.csv",
    index=False
)

In [31]:
for lookback_days, dataset in (lookback_datasets.items()):

    output_dir = (
        model_ready_dir
        / f"lookback_{lookback_days}d"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    np.save(
        output_dir / "X_train_raw.npy",
        dataset["X_train_raw"]
    )

    np.save(
        output_dir / "X_val_raw.npy",
        dataset["X_val_raw"]
    )

    np.save(
        output_dir / "X_test_raw.npy",
        dataset["X_test_raw"]
    )

    np.save(
        output_dir / "X_train_scaled.npy",
        dataset["X_train_scaled"]
    )

    np.save(
        output_dir / "X_val_scaled.npy",
        dataset["X_val_scaled"]
    )

    np.save(
        output_dir / "X_test_scaled.npy",
        dataset["X_test_scaled"]
    )

    np.save(
        output_dir / "y_train.npy",
        dataset["y_train"]
    )

    np.save(
        output_dir / "y_val.npy",
        dataset["y_val"]
    )

    np.save(
        output_dir / "y_test.npy",
        dataset["y_test"]
    )

    dataset["train_metadata"].to_parquet(
        output_dir / "train_metadata.parquet",
        index=False
    )

    dataset[
        "validation_metadata"
    ].to_parquet(
        output_dir
        / "validation_metadata.parquet",
        index=False
    )

    dataset["test_metadata"].to_parquet(
        output_dir / "test_metadata.parquet",
        index=False
    )

    joblib.dump(
        dataset["scaler"],
        output_dir / "feature_scaler.joblib"
    )

    configuration = {
      "asset": "BTCUSDT",
      "candle_interval": "1d",
      "variance_definition": "squared_daily_log_return",
      "lookback_days": lookback_days,
      "forecast_horizon_days": 1,
      "target": "log_realized_variance",
      "feature_columns": FEATURE_COLUMNS,
      "n_features": len(FEATURE_COLUMNS),

      "train_end": str(TRAIN_END),
      "validation_start": str(VALIDATION_START),
      "validation_end": str(VALIDATION_END),
      "test_start": str(TEST_START),
      "test_end": str(TEST_END),

      "X_train_shape": list(
          dataset["X_train_scaled"].shape
      ),
      "X_validation_shape": list(
          dataset["X_val_scaled"].shape
      ),
      "X_test_shape": list(
          dataset["X_test_scaled"].shape
      ),
    }

    with open(
        output_dir
        / "dataset_configuration.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            configuration,
            file,
            indent=2
        )

    print(
        f"Saved lookback {lookback_days}d "
        f"to {output_dir}"
    )

    required_output_files = {
    "X_train_raw.npy",
    "X_val_raw.npy",
    "X_test_raw.npy",
    "X_train_scaled.npy",
    "X_val_scaled.npy",
    "X_test_scaled.npy",
    "y_train.npy",
    "y_val.npy",
    "y_test.npy",
    "train_metadata.parquet",
    "validation_metadata.parquet",
    "test_metadata.parquet",
    "feature_scaler.joblib",
    "dataset_configuration.json",
    }

    saved_output_files = {
        path.name
        for path in output_dir.iterdir()
    }

    missing_files = (
        required_output_files
        - saved_output_files
    )

    if missing_files:
        raise FileNotFoundError(
            f"Missing files for lookback "
            f"{lookback_days}: {missing_files}"
        )

Saved lookback 7d to /content/drive/MyDrive/Quant Research/data/model_ready/BTCUSDT/lookback_7d
Saved lookback 14d to /content/drive/MyDrive/Quant Research/data/model_ready/BTCUSDT/lookback_14d
Saved lookback 30d to /content/drive/MyDrive/Quant Research/data/model_ready/BTCUSDT/lookback_30d


### Final Verification

In [32]:
verification_rows = []

for lookback_days in LOOKBACK_OPTIONS:
    output_dir = (
        model_ready_dir
        / f"lookback_{lookback_days}d"
    )

    loaded_X_train = np.load(
        output_dir / "X_train_scaled.npy"
    )

    loaded_X_val = np.load(
        output_dir / "X_val_scaled.npy"
    )

    loaded_X_test = np.load(
        output_dir / "X_test_scaled.npy"
    )

    loaded_y_train = np.load(
        output_dir / "y_train.npy"
    )

    loaded_y_val = np.load(
        output_dir / "y_val.npy"
    )

    loaded_y_test = np.load(
        output_dir / "y_test.npy"
    )

    loaded_train_metadata = pd.read_parquet(
        output_dir / "train_metadata.parquet"
    )

    loaded_validation_metadata = pd.read_parquet(
        output_dir / "validation_metadata.parquet"
    )

    loaded_test_metadata = pd.read_parquet(
        output_dir / "test_metadata.parquet"
    )

    loaded_scaler = joblib.load(
        output_dir / "feature_scaler.joblib"
    )

    with open(
        output_dir / "dataset_configuration.json",
        "r",
        encoding="utf-8"
    ) as file:
        loaded_configuration = json.load(file)

    assert (
        len(loaded_X_train)
        == len(loaded_y_train)
        == len(loaded_train_metadata)
    )

    assert (
        len(loaded_X_val)
        == len(loaded_y_val)
        == len(loaded_validation_metadata)
    )

    assert (
        len(loaded_X_test)
        == len(loaded_y_test)
        == len(loaded_test_metadata)
    )

    assert loaded_X_train.shape[1] == lookback_days
    assert loaded_X_val.shape[1] == lookback_days
    assert loaded_X_test.shape[1] == lookback_days

    assert loaded_X_train.shape[2] == len(
        FEATURE_COLUMNS
    )

    assert (
        loaded_configuration["lookback_days"]
        == lookback_days
    )

    assert (
        loaded_validation_metadata[
            "target_date"
        ].min()
        >= VALIDATION_START
    )

    assert (
        loaded_validation_metadata[
            "target_date"
        ].max()
        <= VALIDATION_END
    )

    assert (
        loaded_test_metadata[
            "target_date"
        ].min()
        >= TEST_START
    )

    assert (
        loaded_test_metadata[
            "target_date"
        ].max()
        <= TEST_END
    )

    verification_rows.append({
        "lookback_days": lookback_days,
        "train_shape": loaded_X_train.shape,
        "validation_shape": loaded_X_val.shape,
        "test_shape": loaded_X_test.shape,
        "train_metadata_rows": len(
            loaded_train_metadata
        ),
        "validation_metadata_rows": len(
            loaded_validation_metadata
        ),
        "test_metadata_rows": len(
            loaded_test_metadata
        ),
    })

verification_df = pd.DataFrame(
    verification_rows
)

display(verification_df)

print(
    "All lookback datasets saved and "
    "verified successfully."
)

,lookback_days,train_shape,validation_shape,test_shape,train_metadata_rows,validation_metadata_rows,test_metadata_rows
0,7,"(2504, 7, 7)","(365, 7, 7)","(365, 7, 7)",2504,365,365
1,14,"(2497, 14, 7)","(365, 14, 7)","(365, 14, 7)",2497,365,365
2,30,"(2481, 30, 7)","(365, 30, 7)","(365, 30, 7)",2481,365,365


All lookback datasets saved and verified successfully.


In [33]:
for lookback_days, dataset in lookback_datasets.items():

    print(f"\n{'='*50}")
    print(f"LOOKBACK = {lookback_days} DAYS")
    print(f"{'='*50}")

    print(
        "Train:",
        dataset["X_train_scaled"].shape,
        dataset["y_train"].shape
    )

    print(
        "Validation:",
        dataset["X_val_scaled"].shape,
        dataset["y_val"].shape
    )

    print(
        "Test:",
        dataset["X_test_scaled"].shape,
        dataset["y_test"].shape
    )

    assert np.isfinite(
        dataset["X_train_scaled"]
    ).all()

    assert np.isfinite(
        dataset["X_val_scaled"]
    ).all()

    assert np.isfinite(
        dataset["X_test_scaled"]
    ).all()

    assert np.isfinite(
        dataset["y_train"]
    ).all()

    assert np.isfinite(
        dataset["y_val"]
    ).all()

    assert np.isfinite(
        dataset["y_test"]
    ).all()

    assert (
        dataset["sequence_metadata"]["target_date"]
        - dataset["sequence_metadata"]["sequence_end_date"]
    ).eq(
        pd.Timedelta(days=1)
    ).all()

print("\nAll model-ready datasets passed validation.")


LOOKBACK = 7 DAYS
Train: (2504, 7, 7) (2504,)
Validation: (365, 7, 7) (365,)
Test: (365, 7, 7) (365,)

LOOKBACK = 14 DAYS
Train: (2497, 14, 7) (2497,)
Validation: (365, 14, 7) (365,)
Test: (365, 14, 7) (365,)

LOOKBACK = 30 DAYS
Train: (2481, 30, 7) (2481,)
Validation: (365, 30, 7) (365,)
Test: (365, 30, 7) (365,)

All model-ready datasets passed validation.
